# Day 4 — External Variables, the MLP, and a Full Model Synthesis
## Exogenous Predictors, Multilayer Perceptrons, and Memory Across the Week

**Student Learning Outcomes:**
> **SLO 3 (extension):** Build a lag-embedded feature matrix that incorporates
> **external (exogenous) variables** alongside a series' own lagged history.
>
> **SLO 4:** Implement and evaluate forecasting models — including an exogenous-aware
> model and an MLP — using time-aware train/test splits.
>
> **SLO 5:** Quantitatively compare forecasting performance using RMSE and MAE,
> across both univariate and exogenous-aware models.
>
> **SLO 6:** Synthesize how memory works across every model family covered this
> week — from AR's fixed lag window to the LSTM's learned, gated memory.


## Before We Begin

> **Prompt:** Think of something you predict regularly that depends on information
> *outside* its own past — not just "what happened last time," but some other signal
> entirely.

> Write 3–5 sentences describing this prediction and the *external* signal you use.
> Then sketch a simple diagram showing the external signal feeding into your prediction alongside the thing's own past behavior.


---
### Quick Recap from Days 1–3

- **Day 1:** Diagnosed time series — stationarity, transformations, ACF/PACF.
- **Day 2:** Built AR models and the **lag-embedded feature matrix** — re-framing
  forecasting as supervised learning.
- **Day 3:** Implemented six model families (naive, AR, Random Forest, Gradient
  Boosting, MLP, RNN, LSTM) with a **time-aware train/test split** and compared them
  using **RMSE** and **MAE**.

So far, every model has predicted a series using **only its own past values**. But real
forecasting problems often have **external information** available — weather, holidays,
economic indicators — that can improve predictions beyond what the series' own history
tells you.

Today you will:
1. Build a lag matrix that incorporates **external (exogenous) variables**
2. Compare a univariate model to an exogenous-aware model on a real dataset
3. Revisit the MLP, now with exogenous features included
4. Build a complete model comparison table across the whole week's model families
5. Write a capstone reflection connecting every model's notion of "memory"


---
## Part 0 — Imports & Setup

Run the cell below. You do **not** need to modify it.


In [1]:
!pip install -q pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 34.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as L

np.random.seed(42)
torch.manual_seed(42)
L.seed_everything(42)

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

print('All imports successful!')


INFO:lightning_fabric.utilities.seed:Seed set to 42


All imports successful!


---
## Part 1 — A New Dataset: Daily Bike Rentals

Today we switch datasets. The **Capital Bikeshare** daily rental counts (Washington,
D.C., 2011–2012) come with genuine external variables recorded alongside the target:
temperature, humidity, windspeed, and whether the day was a working day or holiday.
This is a much more natural setting for exogenous variables than Air Passengers, which
only ever gave us the series itself.

### 1.1 Load the data

This cell is complete.


In [3]:
url = 'https://raw.githubusercontent.com/christophM/interpretable-ml-book/master/data/bike-sharing-daily.csv'
df = pd.read_csv(url, parse_dates=['dteday'])

# Keep only the columns we need today, with clearer names
df = df[['dteday', 'cnt', 'temp', 'hum', 'windspeed', 'workingday', 'holiday']].copy()
df.columns = ['Date', 'Rentals', 'Temp', 'Humidity', 'Windspeed', 'WorkingDay', 'Holiday']

print('Shape:', df.shape)
df.head()


Shape: (731, 7)


,Date,Rentals,Temp,Humidity,Windspeed,WorkingDay,Holiday
0,2011-01-01,985,0.344167,0.805833,0.160446,0,0
1,2011-01-02,801,0.363478,0.696087,0.248539,0,0
2,2011-01-03,1349,0.196364,0.437273,0.248309,1,0
3,2011-01-04,1562,0.200000,0.590435,0.160296,1,0
4,2011-01-05,1600,0.226957,0.436957,0.186900,1,0


> 💡 **Note on the variables:** `Temp`, `Humidity`, and `Windspeed` are already
> normalized to roughly $[0, 1]$ by the dataset's creators. `WorkingDay` and `Holiday`
> are binary flags (1 = yes, 0 = no). `Rentals` is the total daily bike rental count —
> our target.


### 1.2 Visualize the target and one candidate exogenous variable

**Your turn!** Plot `Rentals` over time, and below it, plot `Temp` over the same
time range. Use two stacked subplots sharing the x-axis.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# FILL IN: plot df['Rentals'] against df['Date'] on the first axis
axes[0].plot(df['???'], df['???'], color='steelblue')
axes[0].set_title('Daily Bike Rentals')

# FILL IN: plot df['Temp'] against df['Date'] on the second axis
axes[1].plot(df['???'], df['???'], color='darkorange')
axes[1].set_title('Normalized Temperature')

plt.tight_layout()
plt.show()


### ✏️ Written Response 1.2

1. Do `Rentals` and `Temp` appear to rise and fall together, or do they move
   independently? Describe what you see.
2. Does this match your intuition about bike rentals and weather? Why might
   temperature be a useful **external** predictor for rental counts?
3. `Rentals` also shows a clear trend across the two years. What might explain that,
   beyond weather alone? *(Hint: this is a bike-share program in its early years.)*

> **YOUR ANSWER:**


### 1.3 Quantify the relationship

This cell is complete, run it to see the correlation between `Rentals` and each
candidate exogenous variable.


In [4]:
exog_candidates = ['Temp', 'Humidity', 'Windspeed', 'WorkingDay', 'Holiday']

correlations = df[['Rentals'] + exog_candidates].corr()['Rentals'].drop('Rentals')
print('Correlation of each candidate variable with Rentals:')
print(correlations.sort_values(ascending=False))


Correlation of each candidate variable with Rentals:
Temp          0.627494
WorkingDay    0.061156
Holiday      -0.068348
Humidity     -0.100659
Windspeed    -0.234545
Name: Rentals, dtype: float64


### ✏️ Written Response 1.3

1. Which variable has the strongest correlation with `Rentals`? Is the sign
   (positive/negative) what you expected?
2. `Humidity` and `Windspeed` typically show weak negative correlations. Propose a
   real-world explanation for why higher humidity or wind might *reduce* rentals.
3. Based on this table alone, which variable would you choose as your primary
   exogenous predictor today?

> **YOUR ANSWER:**


---
## Part 2 — Quick Stationarity Check

Before building any model, run the Day 1 diagnostic habit on the new target series.

**Your turn!** Reuse `run_adf` (rewritten below for convenience) to check whether
`Rentals` is stationary.


In [9]:
from statsmodels.tsa.stattools import adfuller

def run_adf(series, label='Series'):
    result = adfuller(series.dropna())
    print(f'ADF Test: {label}')
    print(f'  p-value: {result[1]:.4f}')
    print('  --> STATIONARY' if result[1] < 0.05 else '  --> NON-STATIONARY')
    print()

# FILL IN: run the ADF test on df['Rentals']
#run_adf(df[???], label='Daily Rentals')


> 💡 **Heads up:** Unlike Air Passengers, the bike rentals series does **not** need
> a log transform — there's no strong multiplicative seasonality here. But it may still
> need differencing if the ADF test says it's non-stationary. If so, apply a first
> difference to `Rentals` and re-test before continuing — exactly the Day 1 workflow,
> just applied to a new series.


In [10]:
# FILL IN: if needed, apply a first difference and re-test
# (If the original series is already stationary, you can skip this and use it directly —
#  just be consistent about which version you use for the rest of the notebook.)
df['Rentals_Diff'] = df['Rentals'].diff()

run_adf(df['Rentals_Diff'], label='Differenced Rentals')


ADF Test: Differenced Rentals
  p-value: 0.0000
  --> STATIONARY



---
## Part 3 — Baseline: Univariate Lag Matrix (No Exogenous Variables)

Before adding external variables, build the same kind of univariate lag matrix from
Day 2 — this is today's baseline for comparison.

**Your turn!** Complete `make_lag_matrix` (you've written this before) and build
$(X, y)$ using only `Rentals_Diff`'s own lags.


In [11]:
def make_lag_matrix(series, n_lags):
    series = np.array(series)
    T = len(series)
    X, y = [], []
    for t in range(n_lags, T):
        X.append(series[t - n_lags : t])
        y.append(series[t])
    return np.array(X), np.array(y)

N_LAGS = 7   # one week of history
target_series = df['Rentals_Diff'].dropna().values

X_uni, y_uni = make_lag_matrix(target_series, N_LAGS)
print('Univariate X shape:', X_uni.shape)


Univariate X shape: (723, 7)


> 💡 **Why `N_LAGS=7`?** Bike rentals are daily data, and weekly patterns (weekday
> vs.\ weekend commuting) are a likely source of short-term dependence. This is a
> different — and equally defensible — choice than the `N_LAGS=12` used for Air
> Passengers' monthly seasonality.


### 3.1 Time-aware split (same rule as Day 3)

**Your turn!** Split chronologically — no shuffling — using the last 20% as test data.


In [12]:
TRAIN_FRAC = 0.80

# FILL IN: compute the split index
split = int(len(X_uni) * TRAIN_FRAC)

X_uni_train, X_uni_test = X_uni[:split], X_uni[split:]
y_uni_train, y_uni_test = y_uni[:split], y_uni[split:]

print(f'Training samples: {len(X_uni_train)}')
print(f'Test samples:     {len(X_uni_test)}')


Training samples: 578
Test samples:     145


### 3.2 Fit a baseline model and evaluate

Use `LinearRegression` (the same AR-equivalent model from Day 2) as today's
univariate baseline.


In [13]:
def compute_metrics(y_true, y_pred, label='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f'{label:<30}  RMSE={rmse:.4f}   MAE={mae:.4f}')
    return {'RMSE': rmse, 'MAE': mae}

results = {}

# FILL IN: fit a LinearRegression on the univariate training data
lr_uni = LinearRegression()
lr_uni.fit(X_uni_train, y_uni_train)
lr_uni_preds = lr_uni.predict(X_uni_test)

results['Univariate (lags only)'] = compute_metrics(y_uni_test, lr_uni_preds, 'Univariate Linear')


Univariate Linear               RMSE=1227.8162   MAE=823.8220


---
## Part 4 — Building an Exogenous-Aware Lag Matrix

### The key idea

On Day 2 you built a *multivariate* lag matrix using lagged values of a second series.
Today's exogenous variables are different in one important way: **we may know today's
weather today** — we don't need to lag it. A forecaster predicting tomorrow's rentals
could plausibly use *today's* temperature reading directly, since it's already observed
by the time the forecast is made.

This gives us a richer feature row:

$$
\mathbf{x}_t = \big[\, \underbrace{y_{t-1}, y_{t-2}, \ldots, y_{t-p}}_{\text{lags of the target}},\ \underbrace{e_t}_{\text{exogenous, same day}} \,\big]
$$

where $e_t$ is the exogenous variable's value on the *same day* as the target $y_t$
we're predicting — not lagged, since it's assumed known in advance (e.g., a weather
forecast).

### 4.1 Build the exogenous-aware lag matrix

**Your turn!** Complete the function below. It should produce the same lag columns as
`make_lag_matrix`, plus one additional column for the chosen exogenous variable —
**aligned to the same time index as the target**, not lagged.


In [14]:
def make_lag_matrix_exog(series, exog, n_lags):
    """
    Build a lag-embedded feature matrix that includes lags of `series` PLUS
    the same-day value of an exogenous variable `exog`.

    Parameters
    ----------
    series : array-like, shape (T,)   the target series
    exog   : array-like, shape (T,)   the exogenous variable, aligned index-for-index
                                        with `series`
    n_lags : int, number of lag features for the target

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags + 1)
    y : np.ndarray, shape (T - n_lags,)
    """
    series = np.array(series)
    exog   = np.array(exog)
    T = len(series)
    X, y = [], []
    for t in range(n_lags, T):
        lag_features = series[t - n_lags : t]
        # FILL IN: get the exogenous value at the SAME time index as the target (t)
        exog_today = exog[t]
        # FILL IN: combine lag_features and exog_today into one row
        # (hint: np.append(lag_features, exog_today))
        row = np.append(lag_features, exog_today)
        X.append(row)
        y.append(series[t])
    return np.array(X), np.array(y)


### 4.2 Apply it using `Temp` as the exogenous variable

**Your turn!** You'll need the exogenous series aligned to the same rows as
`Rentals_Diff` — remember that differencing dropped the first row.


In [15]:
# Align Temp to the same index range as Rentals_Diff (which lost its first row to .diff())
temp_aligned = df['Temp'].iloc[1:].values   # drop first row to match Rentals_Diff

# FILL IN: call make_lag_matrix_exog with target_series, temp_aligned, and N_LAGS
X_exog, y_exog = make_lag_matrix_exog(target_series, temp_aligned, N_LAGS)

print('Exogenous-aware X shape:', X_exog.shape)
print('(Should have one more column than the univariate X_uni.)')


Exogenous-aware X shape: (723, 8)
(Should have one more column than the univariate X_uni.)


### 4.3 Split and fit

**Your turn!** Apply the same chronological split logic as Part 3, then fit a second
`LinearRegression` on the exogenous-aware data.


In [29]:
# FILL IN: chronological split, same TRAIN_FRAC as before
split_exog = int(len(X_exog) * TRAIN_FRAC)

X_exog_train, X_exog_test = X_exog[:split_exog], X_exog[split_exog:]
y_exog_train, y_exog_test = y_exog[:split_exog], y_exog[split_exog:]

lr_exog = LinearRegression()
#lr_exog.fit(???, ???)
#lr_exog_preds = lr_exog.predict(???)

#results['Exogenous (lags + Temp)'] = compute_metrics(y_exog_test, lr_exog_preds, 'Exogenous Linear')


### ✏️ Written Response 4

1. Did adding `Temp` improve RMSE and MAE relative to the univariate model? By how much?
2. Look at `lr_exog.coef_` — the last coefficient corresponds to `Temp`. What sign is
   it, and does that match the correlation you found in Part 1.3?
3. If the improvement was small, propose one reason why temperature alone might not
   add much beyond what the rental series' own recent history already captures.

> **YOUR ANSWER:**


## Part 4.5 - Classes

In [21]:
from pandas.tseries.offsets import Second
class Time:
  """
  represents the time of day.
  attributes: hour, minute, second
  """
  # the init method (initialization) is a special method that is called when an object
  # is instantiated
  def __init__(self, hour=0, minute=0, second=0):
    #assign self.hour as hour
    self.hour = hour
    self.minute = minute
    self.second = second

  def print_time(self):
    print('%.2d:%.2d:%.2d' % (self.hour, self.minute, self.second))


In [24]:
start = Time(1,42,11)

In [25]:
start.print_time()

01:42:11


In [20]:
start.hour

1

---
## Part 5 — The MLP, Revisited With Exogenous Features

Day 3 built an MLP using only lagged values of the target, with a hand-written
training loop (`zero_grad` → forward → loss → `backward` → `step`, repeated for
every epoch). That loop pattern is correct, but you'll write it again and again as
you build more models — and it's easy to introduce a subtle bug (forgetting
`zero_grad()`, forgetting `.eval()` at the right moment) when you're rewriting it
by hand each time.

**Today we introduce PyTorch Lightning** — a thin organizational layer on top of
the exact same PyTorch you already know. Lightning doesn't change *what* your model
computes; it changes *how the training loop gets organized and run*. You already
know what a tensor is and how `nn.Module` works — Lightning builds directly on top
of both.


### 5.1 Scale the exogenous-aware data

**Your turn!** Same scaling rule as Day 3: fit on training data only.


In [30]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# FILL IN: fit_transform on train, transform on test
X_exog_train_s = scaler_X.fit_transform(X_exog_train)
X_exog_test_s  = scaler_X.transform(X_exog_test)

y_exog_train_s = scaler_y.fit_transform(y_exog_train.reshape(-1, 1)).ravel()

X_exog_train_t = torch.tensor(X_exog_train_s, dtype=torch.float32)
y_exog_train_t = torch.tensor(y_exog_train_s, dtype=torch.float32)
X_exog_test_t  = torch.tensor(X_exog_test_s,  dtype=torch.float32)

print('X_exog_train_t shape:', X_exog_train_t.shape)


X_exog_train_t shape: torch.Size([578, 8])


### 5.2 Wrap your tensors in a `Dataset` and `DataLoader`

Lightning expects training data to come from a PyTorch `DataLoader`, which serves
up your data in batches. To build one, you first need a `Dataset` — a small class
that just knows two things: how many examples you have, and how to return the
$i$-th example.

**Your turn!** Complete the `Dataset` below.


In [33]:
class LagDataset(Dataset):
    """
    A minimal PyTorch Dataset wrapping a lag-matrix (X, y) pair.
    """
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        # FILL IN: how many examples are in this dataset?
        return len(self.X)

    def __getitem__(self, idx):
        # FILL IN: return the idx-th feature row and the idx-th target
        return self.X[idx], self.y[idx]


train_dataset = LagDataset(X_exog_train_t, y_exog_train_t)

# FILL IN: wrap train_dataset in a DataLoader. Use batch_size=16, shuffle=True
# (shuffling ROWS within a single training pass is fine — this is not the same as
#  shuffling the train/test SPLIT, which we never do)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print(f'Dataset size: {len(train_dataset)}')
print(f'Number of batches per epoch: {len(train_loader)}')


Dataset size: 578
Number of batches per epoch: 73


> 💡 **Why shuffle rows within an epoch is fine, but shuffling the train/test split
> is not:** `shuffle=True` here only controls the *order* in which already-assigned
> training rows are fed to the model during one pass — every row was already
> determined to belong to the training set by your chronological split back in
> Part 4. No test-set information enters training either way.


### 5.3 Define the model as a `LightningModule`

A `LightningModule` is a regular `nn.Module` with three extra pieces:

- **`forward`** — exactly the same as before: takes input, returns a prediction.
- **`training_step`** — replaces your old hand-written loop body. Given one batch,
  it computes and returns the loss. Lightning handles `zero_grad()`, `backward()`,
  and `step()` for you, automatically, every batch.
- **`configure_optimizers`** — returns the optimizer (same `torch.optim.Adam` you
  already know — nothing new here).

**Your turn!** Complete the class below. The network architecture itself
(`self.net = nn.Sequential(...)`) is identical to Day 3's MLP — only the training
machinery around it changes.


In [34]:
class LitMLPForecaster(L.LightningModule):
    def __init__(self, input_size, hidden_size=32, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def training_step(self, batch, batch_idx):
        # FILL IN: unpack the batch into x and y
        x, y = batch
        # FILL IN: get predictions by calling self on x (same as Day 3's forward pass)
        preds = self(x)
        # FILL IN: compute MSE loss between preds and y
        loss = nn.functional.mse_loss(preds, y)
        self.log('train_loss', loss)   # Lightning logs this automatically each step
        return loss

    def configure_optimizers(self):
        # FILL IN: return an Adam optimizer over self.parameters(), using self.lr
        return torch.optim.Adam(self.parameters(), lr=self.lr)


### 5.4 Train with a `Trainer`

This is the biggest visible change from Day 3: instead of writing a `for epoch in
range(...)` loop yourself, you hand your model and your `DataLoader` to a
`L.Trainer`, and call `.fit()`. The Trainer runs the same forward → loss →
backward → step sequence you wrote by hand on Day 3 — it's just running it for you
now, the same way every time, with far less code.

**Your turn!** Set `max_epochs` and call `.fit()`.


In [35]:
# input_size must be N_LAGS + 1 (lags plus the exogenous column) — same rule as before
mlp_exog = LitMLPForecaster(input_size=N_LAGS + 1, hidden_size=32)

# FILL IN: set max_epochs to something reasonable (Day 3 used 300 manual epochs)
trainer = L.Trainer(max_epochs=100, enable_progress_bar=True, logger=False)

# FILL IN: fit the model using the train_loader you built above
trainer.fit(mlp_exog, train_dataloaders=train_loader)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net  │ Sequential │    833 │ train │     0 │
└───┴──────┴────────────┴────────┴───────┴───────┘

Trainable params: 833                                                                                              
Non-trainable params: 0                                                                                            
Total params: 833                                                                                                  
Total estimated model params size (MB): 0.003                                                                      
Modules in train mode: 6                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


### 5.5 Generate predictions

Prediction works exactly as it did with raw PyTorch: call `.eval()`, wrap in
`torch.no_grad()`, and call the model on your input tensor. Lightning doesn't
change this part at all — `LitMLPForecaster` is still a regular `nn.Module`
underneath everything else.


In [36]:
mlp_exog.eval()
with torch.no_grad():
    mlp_exog_preds_s = mlp_exog(X_exog_test_t).numpy()

# FILL IN: inverse-transform to the original scale
mlp_exog_preds = scaler_y.inverse_transform(mlp_exog_preds_s.reshape(-1,1)).ravel()

results['MLP (lags + Temp)'] = compute_metrics(y_exog_test, mlp_exog_preds, 'MLP Exogenous')


MLP Exogenous                   RMSE=1256.2359   MAE=873.1489


### ✏️ Written Response 5

1. Did the MLP improve on the exogenous-aware linear model? On the univariate model?
2. Identify the three Lightning-specific pieces you added to `LitMLPForecaster`
   (`training_step`, `configure_optimizers`, and the `Dataset`/`DataLoader` setup).
   For each one, name the equivalent line(s) of raw-PyTorch code from Day 3's
   hand-written training loop that it replaces.
3. The MLP can learn *non-linear* combinations of lags and the exogenous variable
   (e.g., "temperature only matters when it's also a working day"). Propose one such
   interaction you think might genuinely exist in this data.
4. Would you expect adding *more* exogenous variables (humidity, windspeed, holiday)
   to keep helping indefinitely, or do you expect diminishing returns? Why?

> **YOUR ANSWER:**


---
## Part 6 — Full Comparison: Univariate vs. Exogenous-Aware

**Your turn!** Assemble everything from today into one sorted comparison table.


In [ ]:
# FILL IN: build a DataFrame from results and sort by RMSE
comparison_df = pd.DataFrame(???).T
comparison_df = comparison_df.sort_values(???)

print('=== Day 4 Model Comparison (sorted by RMSE) ===')
print(comparison_df.to_string(float_format='{:.4f}'.format))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
test_range = np.arange(len(y_exog_test))

ax.plot(test_range, y_exog_test,      label='Actual',                 color='black', linewidth=2)
ax.plot(test_range, lr_uni_preds[-len(y_exog_test):], label='Univariate Linear', linestyle='--')
ax.plot(test_range, lr_exog_preds,    label='Exogenous Linear',       linestyle='-.')
ax.plot(test_range, mlp_exog_preds,   label='MLP (Exogenous)',        color='purple')

ax.set_title('Univariate vs. Exogenous-Aware Forecasts — Test Set')
ax.set_xlabel('Test Step')
ax.legend()
plt.tight_layout()
plt.show()


### ✏️ Written Response 6

1. Rank all models tested today by RMSE. Where does adding the exogenous variable
   help most — the linear model, the MLP, or both equally?
2. If you had to deploy **one** of today's models in a real bike-share system that
   receives a weather forecast each morning, which would you choose, and why?
3. What would you need to verify before trusting this exogenous variable in
   production? *(Hint: think about Part 4's assumption that `Temp` is "known in
   advance." Is that actually true for a forecast, or only for historical data?)*

> **YOUR ANSWER:**


---
## Part 7 — Capstone Reflection: Memory Across the Week

This is a **written-only** section — no new code. Over four days, every model you've
built has answered the same underlying question in a different way: *how much does the
past (and now, the outside world) influence the present?*

| Day | Model | How it represents "memory" |
|---|---|---|
| 2 | AR($p$) | A fixed window of exactly $p$ past values, combined linearly |
| 3 | Random Forest / Gradient Boosting | The same fixed lag window, but combined through non-linear splits |
| 3 | MLP | The same fixed lag window, combined through learned non-linear functions — but lags are treated as an *unordered* set |
| 3 | RNN | A *learned, recursively updated* hidden state — order matters, but long-range influence can vanish |
| 3 | LSTM | A *gated* cell state that can preserve information across many more steps than a plain RNN |
| 4 | Exogenous models | Memory of the series' own past, **plus** same-time information from outside the series entirely |

### ✏️ Final Written Reflection

Write a **6–8 sentence capstone summary**, as if explaining the whole week to someone
who only has tonight to catch up before tomorrow. Your summary must address:

- How the notion of "memory" evolves from AR's fixed window to the LSTM's gated cell
  state — what specifically changes at each step?
- Why a lag-embedded feature matrix was the key idea that made all of this possible
  in the first place
- What an exogenous variable adds that no amount of the series' own lagged history can
  provide
- One concrete recommendation: for a *new* forecasting problem you might encounter,
  what is the first model you'd try, and what would make you reach for something more
  complex?

> **YOUR ANSWER:**


---
## Moving Forward

Choose **one more** exogenous variable from today's dataset (`Humidity`, `Windspeed`,
`WorkingDay`, or `Holiday`) and repeat Parts 4–5 using it instead of `Temp`. Add your
new model's results to the comparison table. Write 2–3 sentences: did this variable
help more or less than `Temp`? Does that match the correlation table from Part 1.3?

```python
# Your code here
```


## References / Further Reading

* [UCI Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)
* [Forecasting: Principles and Practice — Chapter 7 (Regression with ARIMA errors)](https://otexts.com/fpp3/regarima.html)
* [statsmodels SARIMAX documentation](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html)
